# Respiratory virus trends in England from 2020-2026

Use the toggle to switch between the admissions rates and pcr positivity. Each point on the graph represents a month and each line represents the virus. To view the trend of a specific virus, choose which to unselect. Data obtained is from the UKHSA data dashboard. 

Influenza and RSV follow a clear winter pattern, with peaks around November to January. Covid-19 has been less seasonal, with waves at varying times of year.



In [14]:
import json
import os
import datetime
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import ipywidgets as widgets

years = [2020, 2021, 2022, 2023, 2024, 2025, 2026]
england_population = 57000000   # approximate, check the figure for England (ONS)

colours = {"COVID-19": "tab:pink", "Influenza": "tab:blue", "RSV": "tab:purple"}

# everything that differs between the two graphs lives here
measures = {
    "Admissions": {
        "files": {
            "COVID-19": "covid_admissions.json",
            "Influenza": "influenza_admissions.json",
            "RSV": "RSV_cases.json",
        },
        # multiply each value by this to get a weekly rate per 100,000
        "scale": {
            "COVID-19": 7 / england_population * 100000,   # daily count -> weekly rate
            "Influenza": 1,
            "RSV": 1,
        },
        "ylabel": "Average weekly admission rate per 100,000",
        "title": "Monthly hospital admissions, 2020-2026",
    },
    "PCR positivity": {
        # edit these names if your files are called something different
        "files": {
            "COVID-19": "covid_positivity.json",
            "Influenza": "influenza_positivity.json",
            "RSV": "RSV_positivity.json",
        },
        "scale": {"COVID-19": 1, "Influenza": 1, "RSV": 1},   # already a percentage
        "ylabel": "Average PCR positivity (%)",
        "title": "Monthly PCR positivity, 2020-2026",
    },
}

def get_year(record):
    if "year" in record:
        return int(record["year"])
    return int(record["date"][0:4])      # "2023-03-14" -> 2023

def get_month(record):
    if "month" in record:
        return int(record["month"])
    return int(record["date"][5:7])      # "2023-03-14" -> 3

# list of (first day of month, average value) for every month in 2020-2026
def monthly_series(filename, factor):
    with open(filename) as f:
        records = json.load(f)

    dates = []
    values = []
    for year in years:
        for month in range(1, 13):
            total = 0
            count = 0
            for record in records:
                if get_year(record) == year and get_month(record) == month:
                    total = total + record["metric_value"] * factor
                    count = count + 1
            dates.append(datetime.date(year, month, 1))
            if count > 0:
                values.append(total / count)
            else:
                values.append(float("nan"))    # no data -> gap in the line
    return dates, values

# work out every series once: all_series[measure][disease]
all_series = {}
for measure in measures:
    all_series[measure] = {}
    for disease in measures[measure]["files"]:
        filename = measures[measure]["files"][disease]
        if not os.path.exists(filename):
            print("Missing file, skipping:", filename)
            continue
        factor = measures[measure]["scale"][disease]
        all_series[measure][disease] = monthly_series(filename, factor)

# draw the chosen measure for the selected diseases
def draw_chart(measure, selected):
    available = [d for d in selected if d in all_series[measure]]
    if len(available) == 0:
        print("Nothing to show: select a disease that has a data file")
        return

    fig, ax = plt.subplots(figsize=(14, 5))
    for disease in available:
        dates, values = all_series[measure][disease]
        ax.plot(dates, values, marker="o", markersize=3,
                color=colours[disease], label=disease)

    # one label per year, with a light tick for every month
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_minor_locator(mdates.MonthLocator())

    ax.set_xlabel("Year (each point represents a month)")
    ax.set_ylabel(measures[measure]["ylabel"])
    ax.set_title(measures[measure]["title"])
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

measure_toggle = widgets.ToggleButtons(
    options=list(measures),
    value="Admissions",
    description="Show:",
)

# one checkbox per virus, all ticked to start
checkboxes = {}
for disease in colours:
    checkboxes[disease] = widgets.Checkbox(value=True, description=disease)

show_all_button = widgets.Button(description="Show all")

def show_all(button):
    for disease in checkboxes:
        checkboxes[disease].value = True

show_all_button.on_click(show_all)

# collect the ticked viruses and redraw whenever anything changes
def update(measure, **ticks):
    selected = [d for d in ticks if ticks[d]]
    draw_chart(measure, selected)

controls = {"measure": measure_toggle}
for disease in checkboxes:
    controls[disease] = checkboxes[disease]

out = widgets.interactive_output(update, controls)

display(widgets.VBox([
    measure_toggle,
    widgets.HBox(list(checkboxes.values()) + [show_all_button]),
    out,
]))
